In [16]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

df=pd.read_pickle(f"./df.pkl")
y=df[['dm']]
X=df.drop(columns=['dm'])

In [17]:
X.shape

(13796, 25)

In [18]:
X_trainvalid, X_test, y_trainvalid, y_test = train_test_split(
            X, y,
            test_size=0.2,
            random_state=42,
            shuffle=True
        )

In [19]:
X_trainvalid.shape

(11036, 25)

In [20]:
X_train,X_valid, y_train, y_valid=train_test_split(X_trainvalid, y_trainvalid, test_size=0.2, random_state= 65, shuffle=True)

In [21]:
X_valid.shape, y_valid.shape

((2208, 25), (2208, 1))

In [22]:
from model.progression_scoring import progression_scoring
from model.optimize_state import compute_total_decision

import itertools
import pandas as pd
import numpy as np
from pathlib import Path
import pickle

lrs = [0.005, 0.01, 0.02, 0.05]
steps_list = [100, 200, 300, 400, 500]

results=[]

X_eval = X_valid.sample(100, random_state=42)

BASE_DIR = Path.cwd()

model_paths = [
            str(BASE_DIR / "model" / "final_checkpoints" / f"fold_{i}" / "best-checkpoint-v6.ckpt")
            for i in range(5)
        ]

with open(BASE_DIR / "scalers.pkl", "rb") as f:
    scalers = pickle.load(f)
    
def apply_deltas(X, deltas):
        X_opt = X.copy()

        for col, delta in deltas.items():
            if col in X_opt.columns:
                X_opt[col] = X_opt[col] + delta

        return X_opt


for lr, step in itertools.product(lrs,steps_list):

    score_reduction=[]
    l1_changes=[]
    n_changes=[]
    ratio_changes = []
    score_increased = []

    for idx in range(len(X_eval)):

        x = X_eval.iloc[[idx]]

        before = progression_scoring(
            x,
            model_paths,
            scalers
        )

        deltas = compute_total_decision(
            x,
            model_paths,
            scalers,
            epsilon=5,
            lambda_reg=0.2,
            lr=lr,
            steps=step
        )

        x_after = apply_deltas(x,deltas)

        after = progression_scoring(
            x_after,
            model_paths,
            scalers
        )
        
        inv = {
        'wk_smk': (0.0, 420.0),
        'wk_alc': (0.0, 40.0),
        'wk_mvpa_play': (0.0, 300.0),
        'wk_walk': (0.0, 1260.0),
        'wk_sleep': (360.0, 540.0),
        'stress': (1.0, 4.0),
        'wk_break': (0.0, 6.0),
        'wk_lunch': (0.0, 6.0),
        'wk_dinner': (0.0, 6.0),
        'wk_veg1': (0.0, 21.0),
        'wk_veg2': (0.0, 21.0),
        'wk_fruit': (0.0, 21.0),
        }

        ratios=[]
        for var, delta in deltas.items():

            delta = float(delta)
            min_val, max_val = inv[var]
            scale = max_val - min_val

            ratio = abs(delta) / scale
            ratios.append(ratio)



        score_reduction.append(before-after)

        score_increased.append(after > before)

        l1_changes.append(
                np.sum(np.abs(list(deltas.values())))
            )

        n_changes.append(len(deltas))

        ratio_changes.append(np.mean(ratios))

    results.append({

        "lr":lr,
        "step":step,

        "score_reduction":
        np.mean(score_reduction),

        "L1_change":
        np.mean(l1_changes),

        "n_modified":
        np.mean(n_changes),

        "mean_ratio_change": np.mean(ratio_changes),

        "score_increase_rate": np.mean(score_increased),

    })

results=pd.DataFrame(results)

c:\Users\cmc\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\cmc\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\cmc\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\cmc\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\cmc\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning

In [23]:
results

,lr,step,score_reduction,L1_change,n_modified,mean_ratio_change,score_increase_rate
0,0.005,100,2.140838,143.208908,6.32,0.217897,0.01
1,0.005,200,3.031465,242.414280,6.53,0.308127,0.01
2,0.005,300,3.396853,320.284019,6.58,0.358319,0.01
3,0.005,400,3.561799,387.911657,6.58,0.388863,0.01
4,0.005,500,3.715621,423.174749,6.66,0.407177,0.01
5,0.010,100,3.027683,242.447456,6.52,0.308657,0.02
6,0.010,200,3.561137,388.135235,6.64,0.385901,0.01
7,0.010,300,3.811250,462.985996,6.67,0.427684,0.02
8,0.010,400,3.926408,472.726605,6.69,0.453870,0.01
9,0.010,500,4.017668,483.043486,6.72,0.468897,0.02
